In [57]:
#Bloco 0
import pandas as pd
import os
from datetime import timedelta
from genericpath import exists
import shutil

In [58]:
# Bloco 0.1
# Ponto de conexão com o driver
from google.colab import drive, files
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [59]:
#Bloco 2
#Validação da orgiem (arquivo de entrada)
def importar_e_validar_arquivo(origem_entrada):
    """
    Valida a extensão do arquivo, faz a leitura via Pandas e extrai o nome base.
    Retorna uma tupla: (dataframe_origem, nome_processado)
    """
    validacao = False
    # Pega apenas o nome base do arquivo para evitar erro de caminho
    nome = os.path.splitext(os.path.basename(origem_entrada))[0]
    origem = None

    while not validacao:
        # Pega a extensão em letras minúsculas para evitar problemas com .CSV ou .XLSX
        tp_arquivo = os.path.splitext(origem_entrada)[1].lower()
        print(f"Tentando ler extensão: {tp_arquivo}")

        try:
            match tp_arquivo:
                case '.csv':
                    origem = pd.read_csv(origem_entrada)
                    validacao = True

                case '.xlsx' | '.xlsm': # Agrupado usando o operador OR (|)
                    origem = pd.read_excel(origem_entrada)
                    validacao = True

                case '.ods':
                    origem = pd.read_ods(origem_entrada)
                    validacao = True

                case _:
                    print('Formato de arquivo inválido!')

            if validacao:
                print('Success')
                nome = nome.split('_')[0]
            else:
                # Se o formato for inválido, pede um novo caminho para não travar em loop infinito
                origem_entrada = input("Digite o caminho de um arquivo válido (or 'S' para sair): ")
                if origem_entrada.upper() == 'S':
                    return None
                nome = os.path.splitext(os.path.basename(origem_entrada))[0]

        except Exception as erro:
            print(f"Erro ao ler o arquivo: {erro}")
            sair = input("Deu ruim, sair(S) ou tentar novamente(N)? > ")

            if sair.upper() == 'S':
                return None

            # Se não quiser sair, pede para digitar o novo caminho do arquivo
            origem_entrada = input("Digite o novo caminho do arquivo: ")
            nome = os.path.splitext(os.path.basename(origem_entrada))[0]

    return origem, nome

In [60]:
#Bloco 3
#Troca dos nomes para facilitar a consulta final (em um banco de dados)

def renomear_colunas(df):
    """
    Renomeia as colunas do DataFrame para um padrão sem acentos/espaços,
    facilitando a inserção em bancos de dados.
    """
    mapeamento = {
        'Área': 'Area',
        'Estação': 'Estacao',
        'Produto ativo': 'Produto_ativo',
        'Justificativa': 'Justificativa',
        'Tipo Justificativa': 'Tipo_justificativa',
        'Data Inicial': 'Data_inicial',
        'Data Final': 'Data_final',
        'Segundos': 'Tempo_segundos',
        'HH:MM:SS': 'Tempo_total'
    }

    try:
        # O Pandas já renomeia tudo de uma vez usando o dicionário direto
        df_renomeado = df.rename(columns=mapeamento)
        print('Mapeamento de colunas: Success')
        return df_renomeado

    except Exception as erro:
        print(f'Mapeamento de colunas: Unsuccessful. Erro: {erro}')
        return "Concluido sem exito"  # Retorna o dataframe original caso dê algum erro grave

In [61]:
#Bloco 4
#Correção dos dados da coluna Tipo_justificativa
def corrigir_tipo_justificativa(df_renomeado):
    """
    Corrige a coluna 'Tipo_justificativa' para 'Não Programada'
    sempre que a 'Justificativa' for igual a 'Parada   não declarada'.
    """
    try:
        # Cria uma cópia para evitar o aviso 'SettingWithCopyWarning'
        df_corrigido = df_renomeado.copy()

        # Abordagem vetorizada usando .loc (muito mais rápida que .apply)
        filtro = df_corrigido['Justificativa'] == 'Parada   não declarada'
        df_corrigido.loc[filtro, 'Tipo_justificativa'] = 'Não Programada'

        print('Correção de justificativas: Success')
        return df_corrigido

    except Exception as erro:
        print(f'Correção de justificativas: Unsuccessful. Erro: {erro}')

"""Variaveis das planilhas dos meses"""

'Variaveis das planilhas dos meses'

In [62]:
# Bloco 5
# Função de criação da visualização atualizada para mostrar 'Tempo total' em apenas uma linha
def tabela_tempo(df_destino, dicionario_meses):
    df_destino = df_destino.copy()
    df_destino['Data_inicial'] = pd.to_datetime(df_destino['Data_inicial'], errors='coerce')
    df_destino = df_destino.dropna(subset=['Data_inicial'])

    dicionario = []
    for indicie, i in df_destino.iterrows():
        dicionario.append({
            'Dia': i['Data_inicial'].strftime('%d/%m'),
            'Tipo_justificativa': i['Tipo_justificativa'],
            'Tempo': i['Tempo_segundos']
        })

    dicionario = sorted(dicionario, key=lambda x: x['Dia'])
    df_dicionario = pd.DataFrame(dicionario)

    # Separando os dados
    df_p_raw = df_dicionario[df_dicionario['Tipo_justificativa'] == 'Programada']
    df_np_raw = df_dicionario[df_dicionario['Tipo_justificativa'] != 'Programada']

    # Agrupando por dia e somando os segundos
    df_p_agrupado = df_p_raw.groupby('Dia', as_index=False).agg(segundos=('Tempo', 'sum'))
    df_np_agrupado = df_np_raw.groupby('Dia', as_index=False).agg(segundos=('Tempo', 'sum'))

    # Criando colunas de exibição formatadas
    def formatar_df(df):
        if df.empty: return pd.DataFrame(columns=['Dia', 'Tempo', 'segundos', 'dia', 'mes'])
        df['Tempo'] = df['segundos'].apply(lambda s: str(timedelta(seconds=float(s))))
        df['dia'] = df['Dia'].str.split('/').str[0].astype(int)
        df['mes'] = df['Dia'].str.split('/').str[1].astype(int)
        return df.sort_values(by=['mes', 'dia'])

    df_programada_final = formatar_df(df_p_agrupado)
    df_nao_programada_final = formatar_df(df_np_agrupado)

    meses_validos = []
    for num_mes, nome_mes in dicionario_meses.items():
        filtro_p = df_programada_final[df_programada_final['mes'] == num_mes].copy()
        filtro_np = df_nao_programada_final[df_nao_programada_final['mes'] == num_mes].copy()

        if not filtro_p.empty:
            total_seg = float(filtro_p['segundos'].sum())
            filtro_p = filtro_p[['Dia', 'Tempo']].copy()
            filtro_p['Tempo total'] = ''
            filtro_p.loc[filtro_p.index[0], 'Tempo total'] = str(timedelta(seconds=total_seg))
            globals()[f'df_programada_{nome_mes}'] = filtro_p.reset_index(drop=True)

        if not filtro_np.empty:
            total_seg_np = float(filtro_np['segundos'].sum())
            filtro_np = filtro_np[['Dia', 'Tempo']].copy()
            filtro_np['Tempo total'] = ''
            filtro_np.loc[filtro_np.index[0], 'Tempo total'] = str(timedelta(seconds=total_seg_np))
            globals()[f'df_nao_programada_{nome_mes}'] = filtro_np.reset_index(drop=True)

        if not filtro_p.empty or not filtro_np.empty:
            meses_validos.append(nome_mes)

    globals()['meses_validos'] = meses_validos
    print('Processamento concluído.')
    return df_programada_final, df_nao_programada_final

print('ok')

ok


In [63]:
# Bloco 6
# Atualizado com a aba 'Horas totais'
# Criando as novas planilhas

def gerar_relatorio_mensal_justificativas(df, dicionario_meses):
    """
    Agrupa os dados por mês e justificativa, calcula o tempo total em HH:MM:SS,
    cria DataFrames globais por mês e retorna o DataFrame consolidado.
    """
    try:
        # Criando cópia para manipulação e garantindo o formato de data
        df_aux = df.copy()
        df_aux['Data_inicial'] = pd.to_datetime(df_aux['Data_inicial'])
        df_aux['Mes_Num'] = df_aux['Data_inicial'].dt.month

        # Lista para consolidar os dados de todos os meses
        dados_consolidados = []

        print("Agrupamento por Mês e Justificativa (Ordenado por Tempo Total):\n")

        for num_mes, nome_mes in dicionario_meses.items():
            # Filtrando os dados do mês vigente
            df_mes = df_aux[df_aux['Mes_Num'] == num_mes]

            if not df_mes.empty:
                # Agrupando por Justificativa e somando os segundos
                resumo = df_mes.groupby('Justificativa')['Tempo_segundos'].sum().reset_index()

                # Ordenando do maior tempo para o menor
                resumo = resumo.sort_values(by='Tempo_segundos', ascending=False)

                # Convertendo segundos para o formato HH:MM:SS
                resumo['Tempo total'] = resumo['Tempo_segundos'].apply(lambda s: str(timedelta(seconds=float(s))))

                # Removendo a coluna auxiliar de segundos para a visualização e salvamento
                resumo_final = resumo[['Justificativa', 'Tempo total']].copy()

                # Salvando em variável global seguindo a lógica solicitada
                globals()[f'df_justificativa_{nome_mes}'] = resumo_final

                # Adicionando coluna de Mês apenas para o DataFrame consolidado
                resumo_consolidado = resumo_final.copy()
                resumo_consolidado.insert(0, 'Mês', nome_mes.capitalize())
                dados_consolidados.append(resumo_consolidado)

                print(f"--- {nome_mes.capitalize()} (Salvo em df_justificativa_{nome_mes}) ---")
                display(resumo_final.style.hide(axis='index'))
                print("\n")

        # Criando o DataFrame final para exportação
        if dados_consolidados:
            df_consolidado_final = pd.concat(dados_consolidados, ignore_index=True)
            print("Tabela 'df_consolidado_justificativas' pronta para salvar no Excel!")
            return df_consolidado_final
        else:
            print("Nenhum dado encontrado para processar.")


    except Exception as erro:
        print(f"Erro ao processar o agrupamento mensal: {erro}")

def exportar_arquivos_processados(origem_entrada, nome, dicionario_meses):
    """
    Exporta arquivos Excel por mês.
    Corrigido para percorrer nomes de meses, não números, e para usar o mesmo caminho base do processed.
    """
    try:
        ano = None
        for parte in origem_entrada.split(os.sep):
            if parte.isdigit() and int(parte) > 2000:
                ano = int(parte)
                break

        if ano is None:
            print(f'Ano inválido ou não detectado para: {nome}')
            return False

        pasta_processados_base = '/content/drive/MyDrive/bases_projetos_github/tratamento_base_de_paradas/files/processed/mes'
        arquivos_salvos = 0

        for num_mes, mes in dicionario_meses.items():
            sheet_destino = os.path.join(
                pasta_processados_base,
                str(ano),
                nome,
                f'{nome}_{mes}.xlsx'
            )

            df_p = globals().get(f'df_programada_{mes}')
            df_np = globals().get(f'df_nao_programada_{mes}')
            df_j = globals().get(f'df_justificativa_{mes}')

            tem_p = df_p is not None and not df_p.empty
            tem_np = df_np is not None and not df_np.empty
            tem_j = df_j is not None and not df_j.empty

            if not (tem_p or tem_np or tem_j):
                print(f'Sem dados para {mes}; pulando.')
                continue

            resumo_dados = []
            if tem_p:
                total_p = df_p.loc[df_p['Tempo total'].astype(str) != '', 'Tempo total'].iloc[0]
                resumo_dados.append({'Tipo justificativa': 'Programada', 'Total de horas': total_p})

            if tem_np:
                total_np = df_np.loc[df_np['Tempo total'].astype(str) != '', 'Tempo total'].iloc[0]
                resumo_dados.append({'Tipo justificativa': 'Não Programada', 'Total de horas': total_np})

            df_horas_totais = pd.DataFrame(resumo_dados)

            os.makedirs(os.path.dirname(sheet_destino), exist_ok=True)

            with pd.ExcelWriter(sheet_destino, engine='openpyxl') as writer:
                if tem_p:
                    df_p.to_excel(writer, sheet_name='programada', index=False)
                if tem_np:
                    df_np.to_excel(writer, sheet_name='nao_programada', index=False)
                if not df_horas_totais.empty:
                    df_horas_totais.to_excel(writer, sheet_name='Horas totais', index=False)
                if tem_j:
                    df_j.to_excel(writer, sheet_name='Tempo justificativa', index=False)

            arquivos_salvos += 1
            print(f'Arquivo salvo: {sheet_destino}')

        if arquivos_salvos == 0:
            print(f'Nenhuma planilha foi gerada para {nome}. Verifique se há dados nos meses.')
            return False

        return True
    except Exception as erro:
        print(f"Erro ao exportar {nome}: {erro}")
        return False

# Defina o caminho da pasta que deseja explorar
def walker_diretorio(caminho_pasta):
  arquivos_encontrados = []
  # os.walk percorre a pasta e subpastas
  for raiz, diretorios, arquivos in os.walk(caminho_pasta):
      for arquivo in arquivos:
          caminho_completo = os.path.join(raiz, arquivo)

          arquivos_encontrados.append(caminho_completo)
  return arquivos_encontrados

In [64]:
#Bloco de alinhamento

'''
walker_diretorio(caminho_pasta)

importar_e_validar_arquivo(origem_entrada)

renomear_colunas(df_origem)

corrigir_tipo_justificativa(df_renomeado)

gerar_relatorio_mensal_justificativas(df, dicionario_meses)

exportar_arquivos_processados(origem_entrada, nome, lista_meses)
'''

'\nwalker_diretorio(caminho_pasta)\n\nimportar_e_validar_arquivo(origem_entrada)\n\nrenomear_colunas(df_origem)\n\ncorrigir_tipo_justificativa(df_renomeado)\n\ngerar_relatorio_mensal_justificativas(df, dicionario_meses)\n\nexportar_arquivos_processados(origem_entrada, nome, lista_meses)\n'

In [65]:
# Bloco 7
# Exportação consolidada por período completo, sem separar por mês

def formatar_horas(segundos):
    """
    Converte segundos para HH:MM:SS, mesmo quando passa de 24 horas.
    """
    segundos = int(float(segundos))
    horas = segundos // 3600
    minutos = (segundos % 3600) // 60
    segundos_restantes = segundos % 60

    return f"{horas:02d}:{minutos:02d}:{segundos_restantes:02d}"


def gerar_tabelas_periodo(df):
    """
    Gera as tabelas consolidadas do período inteiro:
    - Programada por dia
    - Não programada por dia
    - Horas totais do período
    - Tempo por justificativa no período
    """

    df_aux = df.copy()

    df_aux['Data_inicial'] = pd.to_datetime(df_aux['Data_inicial'])
    df_aux['Dia_Data'] = df_aux['Data_inicial'].dt.date
    df_aux['Dia'] = df_aux['Data_inicial'].dt.strftime('%d/%m/%Y')

    # Programadas por dia

    df_programada = (
        df_aux[df_aux['Tipo_justificativa'] == 'Programada']
        .groupby(['Dia_Data', 'Dia'], as_index=False)
        .agg(segundos=('Tempo_segundos', 'sum'))
        .sort_values('Dia_Data')
    )

    df_programada['Tempo'] = df_programada['segundos'].apply(formatar_horas)
    df_programada_final = df_programada[['Dia', 'Tempo']].copy()

    # Não programadas por dia

    df_nao_programada = (
        df_aux[df_aux['Tipo_justificativa'] != 'Programada']
        .groupby(['Dia_Data', 'Dia'], as_index=False)
        .agg(segundos=('Tempo_segundos', 'sum'))
        .sort_values('Dia_Data')
    )

    df_nao_programada['Tempo'] = df_nao_programada['segundos'].apply(formatar_horas)
    df_nao_programada_final = df_nao_programada[['Dia', 'Tempo']].copy()


    # Horas totais do período

    total_programada = df_programada['segundos'].sum() if not df_programada.empty else 0
    total_nao_programada = df_nao_programada['segundos'].sum() if not df_nao_programada.empty else 0

    df_horas_totais = pd.DataFrame([
        {
            'Tipo justificativa': 'Programada',
            'Total de horas': formatar_horas(total_programada)
        },
        {
            'Tipo justificativa': 'Não Programada',
            'Total de horas': formatar_horas(total_nao_programada)
        },
        {
            'Tipo justificativa': 'Total geral',
            'Total de horas': formatar_horas(total_programada + total_nao_programada)
        }
    ])


    # Tempo por justificativa no período

    df_justificativa = (
        df_aux
        .groupby('Justificativa', as_index=False)
        .agg(segundos=('Tempo_segundos', 'sum'))
        .sort_values('segundos', ascending=False)
    )

    df_justificativa['Tempo total'] = df_justificativa['segundos'].apply(formatar_horas)
    df_justificativa_final = df_justificativa[['Justificativa', 'Tempo total']].copy()

    return df_programada_final, df_nao_programada_final, df_horas_totais, df_justificativa_final


def exportar_arquivo_periodo(origem_entrada, nome, df):
    """
    Exporta uma única planilha do período inteiro.
    Não separa por mês.
    """

    try:
        if '/2025/' in origem_entrada:
            ano = 2025
        elif '/2026/' in origem_entrada:
            ano = 2026
        else:
            ano = 0

        if ano <= 2000:
            print(f'Ano inválido ou não detectado para: {nome}')
            return False

        pasta_destino = f'/content/drive/MyDrive/bases_projetos_github/tratamento_base_de_paradas/files/processed/periodo/{ano}/{nome}'
        os.makedirs(pasta_destino, exist_ok=True)

        caminho_saida = f'{pasta_destino}/{nome}_periodo.xlsx'

        df_programada, df_nao_programada, df_horas_totais, df_justificativa = gerar_tabelas_periodo(df)

        with pd.ExcelWriter(caminho_saida, engine='openpyxl') as writer:

            if not df_programada.empty:
                df_programada.to_excel(writer, sheet_name='programada', index=False)

            if not df_nao_programada.empty:
                df_nao_programada.to_excel(writer, sheet_name='nao_programada', index=False)

            df_horas_totais.to_excel(writer, sheet_name='Horas totais', index=False)

            if not df_justificativa.empty:
                df_justificativa.to_excel(writer, sheet_name='Tempo justificativa', index=False)

            # Opcional: salva a base tratada também
            df.to_excel(writer, sheet_name='Base processada', index=False)

        print(f'Arquivo do período salvo: {caminho_saida}')
        return True

    except Exception as erro:
        print(f'Erro ao exportar período de {nome}: {erro}')
        return False

In [66]:
# Bloco 8
# Dicionario para facilitar a validação e para separar por meses
dicionario_meses = {
        1: "janeiro",
        2: "fevereiro",
        3: "marco",
        4: "abril",
        5: "maio",
        6: "junho",
        7: "julho",
        8: "agosto",
        9: "setembro",
        10: "outubro",
        11: "novembro",
        12: "dezembro"
    }

escolha = input('(M)ês, (P)eriodo ou (T)otal').upper()

if escolha == 'M':

  caminho_pasta = '/content/drive/MyDrive/bases_projetos_github/tratamento_base_de_paradas/files/raw'
  arquivos_encontrados = walker_diretorio(caminho_pasta)
  print(f'Arquivos encontrados: {len(arquivos_encontrados)}')

  for arquivo in arquivos_encontrados:

    for chave in list(globals().keys()):
        if chave.startswith(('df_programada_', 'df_nao_programada_', 'df_justificativa_')):
            del globals()[chave]

    resultado_importacao = importar_e_validar_arquivo(arquivo)
    if resultado_importacao is None:
        print(f'Arquivo ignorado: {arquivo}')
        continue

    origem_entrada, nome = resultado_importacao

    df_renomeado = renomear_colunas(origem_entrada)

    df_corrigido = corrigir_tipo_justificativa(df_renomeado)

    df_programada_final, df_nao_programada_final = tabela_tempo(df_corrigido, dicionario_meses)

    gerar_relatorio_mensal_justificativas(df_corrigido, dicionario_meses)

    exportar_arquivos_processados(arquivo, nome, dicionario_meses)
  print("Processo finalizado")

elif escolha == 'P':
  # Execução do pipeline por período, sem separar por mês
  caminho_pasta = '/content/drive/MyDrive/bases_projetos_github/tratamento_base_de_paradas/files/raw'

  arquivos_encontrados = walker_diretorio(caminho_pasta)

  print(f'Arquivos encontrados: {len(arquivos_encontrados)}')

  for arquivo in arquivos_encontrados:

      resultado = importar_e_validar_arquivo(arquivo)

      if resultado is None:
          print(f'Arquivo ignorado: {arquivo}')
          continue

      origem_entrada, nome = resultado

      df_renomeado = renomear_colunas(origem_entrada)

      df_corrigido = corrigir_tipo_justificativa(df_renomeado)

      exportar_arquivo_periodo(arquivo, nome, df_corrigido)

  print("Processo por período finalizado")

elif escolha == 'T':
  if escolha != 1:
    caminho_pasta = '/content/drive/MyDrive/bases_projetos_github/tratamento_base_de_paradas/files/raw'
    arquivos_encontrados = walker_diretorio(caminho_pasta)
    print(f'Arquivos encontrados: {len(arquivos_encontrados)}')

    for arquivo in arquivos_encontrados:

      for chave in list(globals().keys()):
          if chave.startswith(('df_programada_', 'df_nao_programada_', 'df_justificativa_')):
              del globals()[chave]

      resultado_importacao = importar_e_validar_arquivo(arquivo)
      if resultado_importacao is None:
          print(f'Arquivo ignorado: {arquivo}')
          continue

      origem_entrada, nome = resultado_importacao

      df_renomeado = renomear_colunas(origem_entrada)

      df_corrigido = corrigir_tipo_justificativa(df_renomeado)

      df_programada_final, df_nao_programada_final = tabela_tempo(df_corrigido, dicionario_meses)

      gerar_relatorio_mensal_justificativas(df_corrigido, dicionario_meses)

      exportar_arquivos_processados(arquivo, nome, dicionario_meses)
    print("Processo finalizado")

  if escolha != 2:
    # Execução do pipeline por período, sem separar por mês
    caminho_pasta = '/content/drive/MyDrive/bases_projetos_github/tratamento_base_de_paradas/files/raw'

    arquivos_encontrados = walker_diretorio(caminho_pasta)

    print(f'Arquivos encontrados: {len(arquivos_encontrados)}')

    for arquivo in arquivos_encontrados:

        resultado = importar_e_validar_arquivo(arquivo)

        if resultado is None:
            print(f'Arquivo ignorado: {arquivo}')
            continue

        origem_entrada, nome = resultado

        df_renomeado = renomear_colunas(origem_entrada)

        df_corrigido = corrigir_tipo_justificativa(df_renomeado)

        exportar_arquivo_periodo(arquivo, nome, df_corrigido)

    print("Processo por período finalizado")



(M)ês, (P)eriodo ou (T)otalt
Arquivos encontrados: 1
Tentando ler extensão: .xlsx
Success
Mapeamento de colunas: Success
Correção de justificativas: Success
Processamento concluído.
Agrupamento por Mês e Justificativa (Ordenado por Tempo Total):

--- Janeiro (Salvo em df_justificativa_janeiro) ---


Justificativa,Tempo total
ESPACO RETRABALHO,0:14:36
OPERADOR ATRASADO,0:00:47
FALTA OPERADOR,0:00:41
SETUP,0:00:29
REUNIAO,0:00:20
Parada não declarada,0:00:17




--- Fevereiro (Salvo em df_justificativa_fevereiro) ---


Justificativa,Tempo total
ESPACO RETRABALHO,0:08:35
OPERADOR ATRASADO,0:01:20
TOMBADOR,0:00:53
PRODUTO NÃO CONFORME,0:00:38
FALTA MATERIA PRIMA,0:00:32
FALTA PECA PINTURA,0:00:27




--- Marco (Salvo em df_justificativa_marco) ---


...



--- Maio (Salvo em df_justificativa_maio) ---


Justificativa,Tempo total
ESPACO RETRABALHO,0:10:44
PRODUTO NÃO CONFORME,0:08:04
Parada não declarada,0:04:06
OPERADOR ATRASADO,0:03:36
PRODUTO NÃO CONFORME EXPORTAÇÃO,0:01:32
ABASTECIMENTO ALMOXARIFADO,0:01:04
REBITADEIRA,0:00:17




--- Junho (Salvo em df_justificativa_junho) ---


Justificativa,Tempo total
ESPACO RETRABALHO,0:10:11
OPERADOR ATRASADO,0:02:05
PRODUTO NÃO CONFORME,0:00:56
SETUP,0:00:32




Tabela 'df_consolidado_justificativas' pronta para salvar no Excel!
Arquivo salvo: /content/drive/MyDrive/bases_projetos_github/tratamento_base_de_paradas/files/processed/mes/2026/base/base_janeiro.xlsx
Arquivo salvo: /content/drive/MyDrive/bases_projetos_github/tratamento_base_de_paradas/files/processed/mes/2026/base/base_fevereiro.xlsx
Arquivo salvo: /content/drive/MyDrive/bases_projetos_github/tratamento_base_de_paradas/files/processed/mes/2026/base/base_marco.xlsx
Arquivo salvo: /content/drive/MyDrive/bases_projetos_github/tratamento_base_de_paradas/files/processed/mes/2026/base/base_abril.xlsx
Arquivo salvo: /content/drive/MyDrive/bases_projetos_github/tratamento_base_de_paradas/files/processed/mes/2026/base/base_maio.xlsx
Arquivo salvo: /content/drive/MyDrive/bases_projetos_github/tratamento_base_de_paradas/files/processed/mes/2026/base/base_junho.xlsx
Sem dados para julho; pulando.
Sem dados para agosto; pulando.
Sem dados para setembro; pulando.
Sem dados para outubro; puland

In [67]:
# Bloco 9
# Compactação da pasta para formato .zip e download
ex = input('>>>')

if ex == '':

  # Caminho da pasta que você quer baixar
  caminho_pasta = '/content/drive/MyDrive/bases_projetos_github/tratamento_base_de_paradas/files/processed'
  nome_arquivo_zip = 'Paradas_processed'

  # Compactando a pasta
  shutil.make_archive(nome_arquivo_zip, 'zip', caminho_pasta)

  # Fazendo o download do arquivo zip
  files.download(f'{nome_arquivo_zip}.zip')

>>>


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>